# Datenaufbereitung für die semantische Satzähnlichkeit 

Wenn wir Textdaten abrufen möchten, benötigen wir Textdokumente. Für diesen Kurs nutzen wir die [Allgemeine Debatte der 78. Sitzung der Generalversammlung der Vereinten Nationen](https://en.wikipedia.org/wiki/General_debate_of_the_seventy-eighth_session_of_the_United_Nations_General_Assembly) ausgewählt, die kostenlos und online verfügbar ist. In den allgemeinen Debatten kann jedes Land eine (lange) Rede halten.

Leider gibt es bei diesen Dokumenten einige Probleme:
* Manchmal wurde „.“ an falschen Stellen verwendet. Ich habe versucht, das zu korrigieren.
* Die Debatten sind lang, das Abrufen der richtigen Debatte reicht nicht aus.
* Daher müssen wir mit kleineren Einheiten arbeiten.

In diesem ersten Notebook sind die Debatten in Sätze unterteilt. Normalerweise würde man dann mehrere Sätze zu Fragmenten zusammenfassen (Chunking). In diesem Fall sind die Sätze jedoch bereits recht lang und in sich geschlossen. Daher behalten wir die Sätze als Einheiten bei.

Die Satzsegmentierung klingt einfach, aber wir können nicht einfach einen `.` verwenden, um die Sätze zu trennen. Das würde bei „Mr. X“ oder ähnlichen Konstruktionen nicht funktionieren. Stattdessen nutzen wir [spacy](https://spacy.io) als Tool für die linguistische Analyse, das auch diese Satzsegmentierung durchführen kann.

Dieses Notebook lädt die Daten, teilt die Sätze auf und speichert sie im `json`-Format, damit wir sie in den nachfolgenden Notebooks verwenden können.

## Load data

In [ ]:
import glob
import os

In [ ]:
data = []
for n in glob.glob("un/TXT/Session 78 - 2023/*.txt"):
    data.append({"country": os.path.basename(n.replace("_78_2023.txt", "")), "text": open(n).read() })

In [ ]:
import pandas as pd

In [ ]:
pd.set_option('display.max_colwidth', 500)
df = pd.DataFrame(data)
df

## Sentence segmentation

In [ ]:
import spacy

Jede Sprache verfügt über mehrere Modelle, die zur Textanalyse verwendet werden können. In diesem Fall verwenden wir ein kleines Modell für Englisch, da wir uns ausschließlich für die Satzsegmentierung interessieren. Wären wir an der Erkennung benannter Entitäten interessiert, wäre ein größeres Modell besser geeignet.

In [ ]:
%python -m spacy download en_core_web_sm

In [ ]:
nlp = spacy.load("en_core_web_sm")

In [ ]:
from tqdm.auto import tqdm

In [ ]:
# läuft ~ 1min
sentences = []
for text in tqdm(df["text"]):
    doc = nlp(text)
    for sentence in doc.sents:
        sentences.append(str(sentence).strip())

In [ ]:
sentences[0:20]

In [ ]:
len(sentences)

In [ ]:
import json
with open("sentences.json", "w") as f:
    f.write(json.dumps(sentences))